# Notebook 02: The Agent's Mind — Memory & RAG

An agent without memory forgets everything after each turn. Memory is what lets agents learn across interactions, recall relevant facts, and build persistent knowledge.

**Expected time:** ~60 minutes
**Prerequisites:** Notebooks 00-01


## What You'll Learn

1. Short-term memory: the context window and its limits
2. Running summary: compress history to fit within context
3. RAG (Retrieval-Augmented Generation): embed, store, retrieve, answer
4. Domain adaptation: switch document corpora with a config constant
5. Common memory failures: chunk mismatch, irrelevant retrieval, context overflow


## Setup


In [1]:
!pip install openai chromadb langchain langchain-openai langchain-groq langgraph sentence-transformers


  Using cached openai-2.45.0-py3-none-any.whl.metadata (34 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.13.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (38 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached cuda_bindings-13.3.1-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.5 kB)
  Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_nccl_cu13-2.29.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cuda_nvrtc-13.0.88-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime-13.0.96-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metada

In [2]:
import os
from pathlib import Path
import getpass
import json

from agent_helpers import Memory
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage

# --- Optional .env file loading (no extra pip installs needed) ---
env_path = Path.cwd() / ".env"
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
    print("Loaded .env file.")
else:
    print("No .env file found — you will be prompted for API keys.")
    print("  Tip: cp .env.example .env  and fill in your keys to skip prompts.")

# --- Pick your provider ---
PROVIDER = "openrouter"  # Choose: groq, openrouter
# --------------------------

MOCK = {
    "groq":       True,
    "openrouter": False,
}

# Environment variable names for auto-loading keys (set in .env)
KEY_ENV_VAR = {
    "groq":       "GROQ_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
}

CONFIG = {
    "groq": {
        "name": "Groq",
        "model": "openai/gpt-oss-20b",
        "key_url": "https://console.groq.com/keys",
    },
    "openrouter": {
        "name": "OpenRouter",
        "model": "nvidia/nemotron-3-ultra-550b-a55b:free",
        "key_url": "https://openrouter.ai/keys",
    },
}

cfg = CONFIG[PROVIDER]
use_mock = MOCK[PROVIDER]

if use_mock:
    print(f"[{cfg['name']}] Mock mode enabled. Using canned responses.")
    llm = None
else:
    if PROVIDER == "groq":
        from langchain_groq import ChatGroq
        api_key = os.environ.get(KEY_ENV_VAR[PROVIDER]) or getpass.getpass(f"Enter your {cfg['name']} API key: ")
        llm = ChatGroq(api_key=api_key, model=cfg["model"])
    elif PROVIDER == "openrouter":
        from langchain_openai import ChatOpenAI
        api_key = os.environ.get(KEY_ENV_VAR[PROVIDER]) or getpass.getpass(f"Enter your {cfg['name']} API key: ")
        llm = ChatOpenAI(
            api_key=api_key,
            base_url="https://openrouter.ai/api/v1",
            model=cfg["model"],
            default_headers={"HTTP-Referer": "http://localhost", "X-Title": "BEST Course"},
        )
    print(f"[{cfg['name']}] API key configured.")


def llm_call(messages):
    """Send dict messages to the LLM, return text response.

    Converts legacy dict-style messages to LangChain message objects.
    """
    lc_msgs = []
    for m in messages:
        role, content = m["role"], m["content"]
        if role == "system":
            lc_msgs.append(SystemMessage(content=content))
        elif role == "user":
            lc_msgs.append(HumanMessage(content=content))
    return llm.invoke(lc_msgs).content


def simple_rag(query, documents, k=3):
    """Keyword-based retrieval (no embeddings, works offline)."""
    query_words = set(query.lower().split())
    scored = []
    for i, doc in enumerate(documents):
        doc_words = set(doc.lower().split())
        score = len(query_words & doc_words)
        scored.append((score, i, doc))
    scored.sort(key=lambda x: (-x[0], x[1]))
    top = [doc for _, _, doc in scored[:k]]
    return "\n---\n".join(top)


print("\nNOTE: For RAG cells requiring embeddings, mock mode uses keyword overlap.")


/home/filpa/.local/share/mamba/envs/best_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded .env file.
[OpenRouter] API key configured.

NOTE: For RAG cells requiring embeddings, mock mode uses keyword overlap.


## The DOMAIN Config

Many cells below use a `DOMAIN` constant. Set it to your field and the notebook loads relevant example documents. This makes every exercise directly applicable to your PBL project.

Available domains: `biotech`, `geography`, `medicine`, `physics`, `safety`, `engineering`, `math`, `default`

In [3]:
# --- Set your domain ---
DOMAIN = "default"
# Available: biotech, geography, medicine, physics, safety, engineering, math, default
# -----------------------

domain_docs = {
    "default": [
        "Agentic AI systems use a perception-reasoning-action loop to achieve goals.",
        "Memory in AI can be short-term (context window) or long-term (vector stores).",
        "RAG retrieves relevant documents and injects them into the prompt.",
        "The ReAct pattern interleaves reasoning traces with tool calls.",
        "Chunk size affects retrieval quality: too large loses precision, too small loses context.",
        "Embeddings convert text into numerical vectors for similarity search.",
        "Agentic AI operates at different complexity levels from simple tool use to multi-agent systems.",
    ],
    "biotech": [
        "CRISPR-Cas9 enables precise genome editing by targeting specific DNA sequences.",
        "Protein structure prediction tools like AlphaFold have transformed drug discovery.",
        "Single-cell RNA sequencing reveals heterogeneity in cell populations.",
        "Biomolecule design uses computational models to engineer novel proteins.",
        "Literature review automation is critical due to rapid publication growth in biotech.",
    ],
    "geography": [
        "GIS spatial analysis combines satellite imagery with demographic data.",
        "Media narrative analysis detects framing bias across news sources.",
        "Geopolitical risk assessment models monitor trade, conflict, and policy shifts.",
        "Disinformation campaigns often target regional media ecosystems.",
        "Scenario exploration tools help policymakers evaluate intervention outcomes.",
    ],
    "medicine": [
        "Adaptive radiotherapy adjusts treatment parameters based on daily imaging.",
        "Cognitive load monitoring uses physiological signals to detect burnout risk.",
        "Clinical decision support systems integrate patient data with medical literature.",
        "Medical literature synthesis requires filtering thousands of publications.",
        "Patient outcome prediction models use historical treatment data.",
    ],
    "physics": [
        "Reinforcement learning optimizes control parameters in experimental physics.",
        "Real-time adaptive systems adjust instrument settings based on measurements.",
        "Numerical methods in physics require iterative refinement and error checking.",
        "Differential equation solvers benefit from automated parameter tuning.",
        "Radiation dose optimization balances treatment efficacy with tissue safety.",
    ],
    "safety": [
        "Risk assessment frameworks evaluate probability and impact of system failures.",
        "Safety-critical systems require multiple redundant verification layers.",
        "Autonomous system certification needs validated behavior in edge cases.",
        "Human oversight remains essential for high-stakes AI decisions.",
        "Model-based safety assessment uses formal methods to verify constraints.",
    ],
    "engineering": [
        "Industrial automation systems integrate sensors, actuators, and control logic.",
        "Robotic arm precision depends on inverse kinematics and real-time feedback.",
        "Predictive maintenance uses sensor data to forecast equipment failure.",
        "Autonomous systems in manufacturing require robust error recovery.",
        "Human-robot collaboration demands real-time safety monitoring.",
    ],
    "math": [
        "Pattern discovery algorithms identify structure in high-dimensional data.",
        "Information navigation systems help researchers find relevant mathematical results.",
        "Linear algebra foundations underpin most machine learning algorithms.",
        "Statistical inference quantifies uncertainty in data-driven conclusions.",
        "Optimization methods balance exploration and exploitation in search spaces.",
    ],
}

docs = domain_docs.get(DOMAIN, domain_docs["default"])
print(f"Domain: {DOMAIN}")
print(f"Loaded {len(docs)} documents.")
for d in docs:
    print(f"  \u2022 {d[:70]}...")


Domain: default
Loaded 7 documents.
  • Agentic AI systems use a perception-reasoning-action loop to achieve g...
  • Memory in AI can be short-term (context window) or long-term (vector s...
  • RAG retrieves relevant documents and injects them into the prompt....
  • The ReAct pattern interleaves reasoning traces with tool calls....
  • Chunk size affects retrieval quality: too large loses precision, too s...
  • Embeddings convert text into numerical vectors for similarity search....
  • Agentic AI operates at different complexity levels from simple tool us...


## 1. Short-Term Memory (Context Window)

Every LLM call has a context window limit. For `gpt-4o-mini`, that's 128K tokens. But if you keep adding messages, you WILL hit it.

Let's see what happens when we keep feeding the conversation without managing memory.

In [4]:
def count_tokens(text):
    """Rough token count (words / 0.75)."""
    return int(len(text.split()) / 0.75)

conversation = []
for i in range(5):
    msg = f"Message {i+1}: " + "data " * 1000 + f"(end {i+1})"
    conversation.append({"role": "user", "content": msg})
    total = sum(count_tokens(m["content"]) for m in conversation)
    print(f"After message {i+1}: ~{total} tokens in context")

print("\nA real 128K context fills fast with long tool outputs.")
print("Memory management is how we stay within limits.")


After message 1: ~1338 tokens in context
After message 2: ~2676 tokens in context
After message 3: ~4014 tokens in context
After message 4: ~5352 tokens in context
After message 5: ~6690 tokens in context

A real 128K context fills fast with long tool outputs.
Memory management is how we stay within limits.


## 2. Running Summary Pattern

Instead of keeping every message, compress older ones into a summary. This reduces context from O(N) to O(1).

The `Memory` class in `agent_helpers` does this automatically.

In [5]:
memory = Memory(max_messages=4)

memory.add("user", "What is agentic AI?")
memory.add("assistant", "Agentic AI refers to systems that pursue goals autonomously.")
memory.add("user", "How does it differ from regular AI?")
memory.add("assistant", "Regular AI responds once. Agentic AI loops: plan, act, observe, adapt.")
memory.add("user", "What tools does it need?")

print("Current context (after auto-summarization):")
for m in memory.get_context():
    print(f"  {m['role']}: {m['content'][:70]}...")
print("\nOlder messages compressed into summary. Context stays bounded.")


Current context (after auto-summarization):
  system: Summary of earlier conversation: Agentic AI refers to systems that pur...
  user: How does it differ from regular AI?...
  assistant: Regular AI responds once. Agentic AI loops: plan, act, observe, adapt....
  user: What tools does it need?...

Older messages compressed into summary. Context stays bounded.


## 3. RAG — Retrieval-Augmented Generation

RAG lets an agent access knowledge beyond its training data:

```
Documents -> Split -> Embed -> Store -> Retrieve -> Inject -> Answer
```

We'll use ChromaDB (in-memory vector store) and OpenAI embeddings.

### Step 1: Load and Split Documents


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.create_documents(docs)

print(f"Split {len(docs)} documents into {len(chunks)} chunks.")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {c.page_content[:60]}...")


Split 7 documents into 7 chunks.
  Chunk 0: Agentic AI systems use a perception-reasoning-action loop to...
  Chunk 1: Memory in AI can be short-term (context window) or long-term...
  Chunk 2: RAG retrieves relevant documents and injects them into the p...
  Chunk 3: The ReAct pattern interleaves reasoning traces with tool cal...
  Chunk 4: Chunk size affects retrieval quality: too large loses precis...
  Chunk 5: Embeddings convert text into numerical vectors for similarit...
  Chunk 6: Agentic AI operates at different complexity levels from simp...


### Step 2: Create Embeddings and Vector Store

ChromaDB is zero-config — install and run. No server needed.

In [7]:
USE_PRE_COMPUTED = use_mock

if not USE_PRE_COMPUTED and llm is not None:
    from chromadb import Client
    from chromadb.config import Settings
    from chromadb.utils import embedding_functions

    chroma_client = Client(Settings(persist_directory="./chroma_db", anonymized_telemetry=False))
    try:
        chroma_client.delete_collection("domain_docs")
    except:
        pass

    # Try sentence-transformers first, fall back to ONNX-based default
    try:
        embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        print("Using SentenceTransformer embeddings.")
    except (ImportError, ValueError):
        print("SentenceTransformer not available, falling back to ONNX DefaultEmbeddingFunction.")
        embedding_fn = embedding_functions.DefaultEmbeddingFunction()

    collection = chroma_client.create_collection(
        name="domain_docs",
        embedding_function=embedding_fn,
    )

    collection.add(
        documents=[c.page_content for c in chunks],
        ids=[f"doc_{i}" for i in range(len(chunks))],
    )
    print(f"Vector store created with {len(chunks)} chunks.")
else:
    print("Mock mode — using keyword-based retrieval.")
    collection = None


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Using SentenceTransformer embeddings.
Vector store created with 7 chunks.


### Step 3: Retrieve and Answer

Query the store for relevant chunks, then inject them into the prompt.

In [8]:
def retrieve(query, collection, chunks, k=2):
    """Retrieve top-k relevant chunks."""
    if collection is not None:
        results = collection.query(query_texts=[query], n_results=k)
        return results["documents"][0]
    else:
        return simple_rag(query, [c.page_content for c in chunks], k=k).split("\n---\n")

query = "How do agentic AI systems use memory?"
retrieved = retrieve(query, collection, chunks, k=2)

print(f"Query: {query}\n")
print("Retrieved chunks:")
for i, doc in enumerate(retrieved, 1):
    print(f"  [{i}] {doc[:100]}...")

context = "\n".join(retrieved)
if not use_mock and llm is not None:
    response = llm_call([
        {"role": "system", "content": "Answer using only the provided context."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"},
    ])
    print("\nLLM Answer:")
    print(response)
else:
    print("\nLLM Answer (mock):")
    print("Agentic AI systems use short-term and long-term memory.")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: How do agentic AI systems use memory?

Retrieved chunks:
  [1] Memory in AI can be short-term (context window) or long-term (vector stores)....
  [2] Agentic AI systems use a perception-reasoning-action loop to achieve goals....

LLM Answer:
Based on the provided context, agentic AI systems use a perception-reasoning-action loop to achieve goals, and AI memory exists in two forms: short-term (context window) and long-term (vector stores). The context does not specify how these memory types integrate into the agentic loop.


### Step 4: Full RAG Pipeline (One Function)


In [9]:
def rag_pipeline(query, collection, chunks, k=3):
    """Full RAG pipeline: retrieve -> inject -> answer."""
    retrieved = retrieve(query, collection, chunks, k=k)
    context = "\n".join(retrieved)
    answer = llm_call([
        {"role": "system", "content": "Answer using only the provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ])
    return answer, context

print("rag_pipeline defined. Ready for queries.")


rag_pipeline defined. Ready for queries.


### Try Your Own Query


In [10]:
# --- Write your query ---
my_query = "What are the key challenges in agentic AI?"
# -------------------------

answer, context = rag_pipeline(my_query, collection, chunks)
print(f"Query: {my_query}\n")
print("Retrieved context:")
for i, line in enumerate(context.split("\n"), 1):
    print(f"  [{i}] {line[:80]}...")
print(f"\nAnswer:\n{answer}")


Query: What are the key challenges in agentic AI?

Retrieved context:
  [1] Agentic AI operates at different complexity levels from simple tool use to multi...
  [2] Agentic AI systems use a perception-reasoning-action loop to achieve goals....
  [3] Memory in AI can be short-term (context window) or long-term (vector stores)....

Answer:
The provided context does not mention any key challenges in agentic AI. It only describes complexity levels, the perception-reasoning-action loop, and memory types.


## 4. Advanced RAG: Query Rewriting

A common problem: the user's query is too vague for good retrieval. Solution: have the LLM rewrite the query into a better search query first.

In [11]:
def rewrite_query(user_query):
    """Rewrite a vague query into a specific search query."""
    rewritten = llm_call([{"role": "user", "content": (
        f"Rewrite this vague query into a specific, search-friendly query.\n"
        f"Original: {user_query}\nRewritten:"
    )}])
    return rewritten

vague = "tell me about AI things"
if not use_mock and llm is not None:
    specific = rewrite_query(vague)
else:
    specific = "What are the different types of memory in agentic AI systems?"

print(f"Original: {vague}")
print(f"Rewritten: {specific}")
answer, context = rag_pipeline(specific, collection, chunks)
print(f"\nAnswer:\n{answer}")


Original: tell me about AI things
Rewritten: Here are a few specific, search-friendly versions depending on what you are actually looking for:

**If you want a general overview:**
> **"Artificial Intelligence overview: definition, history, key concepts (ML, DL, NLP), and current applications"**

**If you want to know how it is used right now:**
> **"Real-world applications of Artificial Intelligence in 2024 by industry (healthcare, finance, manufacturing, creative arts)"**

**If you want to understand the technology types:**
> **"Difference between Artificial Intelligence, Machine Learning, Deep Learning, and Generative AI with examples"**

**If you are looking for tools to use:**
> **"Top Generative AI tools 2024 comparison: ChatGPT, Claude, Gemini, Midjourney, Runway (use cases, pricing, pros/cons)"**

**If you are interested in the risks/ethics:**
> **"Current ethical concerns and risks of AI: bias, hallucinations, copyright, job displacement, and regulatory landscape (EU AI Act, US

## Memory in the Agent Loop

Combine memory with the agent loop from Notebook 01. The agent uses `Memory` to persist context across tool calls.

In [12]:
@tool
def search_docs(query: str) -> str:
    """Search the domain knowledge base for relevant information."""
    return simple_rag(query, docs, k=2)


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Provide a valid Python expression like 2+2."""
    return str(eval(expression))


tools = [search_docs, calculator]

if not use_mock and llm is not None:
    agent = create_react_agent(
        model=llm,
        tools=tools,
        prompt="You are a helpful assistant. Use tools when needed to provide accurate answers.",
    )
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Search for information about agent memory"}]}
    )
    final_msg = result["messages"][-1]
    print("\nAgent result:")
    print(final_msg.content)
else:
    print("Mock — agent with memory tools:")
    print("  1. Agent calls search_docs to find info about memory")
    print("  2. Agent combines results into final answer")


/tmp/ipykernel_1423/3040358131.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



Agent result:
Based on the search results, here's what I found about agent memory:

**Agent Memory in AI Systems:**

1. **Short-term Memory (Context Window):**
   - This is the immediate working memory that holds recent conversation history, current task context, and intermediate reasoning steps
   - Limited by the model's context window size
   - Used for maintaining coherence within a single conversation or task execution

2. **Long-term Memory (Vector Stores):**
   - Persistent storage for knowledge, experiences, and learned information across sessions
   - Typically implemented using vector databases (embeddings) for semantic search and retrieval
   - Allows agents to "remember" past interactions, facts, and patterns over extended periods

**In Agentic AI Systems:**
Memory works within the **perception-reasoning-action loop** where:
- **Perception**: Agent gathers information from environment/input
- **Reasoning**: Agent processes information using both short-term context and retr

## What Breaks?


### Break 1: Chunk Size Mismatch


In [13]:
test_doc = "Agentic AI uses planning to decompose tasks. " * 20

for size, overlap in [(50, 5), (500, 50)]:
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    c = s.create_documents([test_doc])
    print(f"Chunk size {size}: {len(c)} chunks")
    if c:
        print(f"  First chunk: {c[0].page_content[:60]}...")


Chunk size 50: 20 chunks
  First chunk: Agentic AI uses planning to decompose tasks....
Chunk size 500: 2 chunks
  First chunk: Agentic AI uses planning to decompose tasks. Agentic AI uses...


### Break 2: Irrelevant Retrieval


In [14]:
bad_query = "What is the capital of France?"
results = retrieve(bad_query, collection, chunks, k=2)
print(f"Query: {bad_query}")
print("Retrieved (likely irrelevant):")
for r in results:
    print(f"  {r[:70]}...")
print("\nFix: Add a relevance check before answering.")


Query: What is the capital of France?
Retrieved (likely irrelevant):
  Agentic AI systems use a perception-reasoning-action loop to achieve g...
  Agentic AI operates at different complexity levels from simple tool us...

Fix: Add a relevance check before answering.


### Break 3: Too Many Chunks


In [15]:
k = 10
results = retrieve("agentic AI", collection, chunks, k=k)
total_chars = sum(len(r) for r in results)
print(f"Retrieved {k} chunks: ~{total_chars} chars (~{total_chars // 4} tokens)")
print("\nFix: Limit k to 3-5. Prioritize relevance over quantity.")


Number of requested results 10 is greater than number of elements in index 7, updating n_results = 7


Retrieved 10 chunks: ~534 chars (~133 tokens)

Fix: Limit k to 3-5. Prioritize relevance over quantity.


## Connection to Agentic AI

Memory separates stateless chatbots from stateful agents.
- **Short-term**: current task context (what step am I on?)
- **Long-term (RAG)**: external knowledge (what do I know about this topic?)

**Complexity Ladder:**
- Level 1: No memory — single-turn tool call
- Level 2: Running summary — remembers across turns
- Level 3-4: RAG + persistent vector store


### PBL Reflection


In [16]:
print("Think about YOUR project:")
print("  1. What knowledge does your agent need outside the LLM's training data?")
print("  2. Short-term summary enough, or need long-term RAG?")
print("  3. What are your documents? PDFs, DB records, web pages?")
print("  4. What if retrieval returns irrelevant results?")


Think about YOUR project:
  1. What knowledge does your agent need outside the LLM's training data?
  2. Short-term summary enough, or need long-term RAG?
  3. What are your documents? PDFs, DB records, web pages?
  4. What if retrieval returns irrelevant results?


## Exercises


### Bronze — Run RAG on Your Domain

Set `DOMAIN` above to your field, then run RAG with a project-relevant query.

In [17]:
print(f"Current domain: {DOMAIN}")
for d in docs:
    print(f"  \u2022 {d[:60]}...")
query = input("Enter a question about your domain: ")
answer, context = rag_pipeline(query, collection, chunks)
print(f"\n--- Context ---\n{context[:200]}...\n")
print(f"--- Answer ---\n{answer}")


Current domain: default
  • Agentic AI systems use a perception-reasoning-action loop to...
  • Memory in AI can be short-term (context window) or long-term...
  • RAG retrieves relevant documents and injects them into the p...
  • The ReAct pattern interleaves reasoning traces with tool cal...
  • Chunk size affects retrieval quality: too large loses precis...
  • Embeddings convert text into numerical vectors for similarit...
  • Agentic AI operates at different complexity levels from simp...


Enter a question about your domain:  What is the difference between embedding and vector?



--- Context ---
Embeddings convert text into numerical vectors for similarity search.
Memory in AI can be short-term (context window) or long-term (vector stores).
Chunk size affects retrieval quality: too large lose...

--- Answer ---
Based on the provided context, embeddings are the mechanism that "convert text into numerical vectors." The vector is the resulting numerical representation used for similarity search.


### Silver — Tune Chunk Size and k

Compare 3 configurations. Which works best for your domain?

In [18]:
configs = [
    {"size": 100, "overlap": 10, "k": 1},
    {"size": 200, "overlap": 20, "k": 2},
    {"size": 500, "overlap": 50, "k": 5},
]
q = "memory in agentic AI"
for cfg in configs:
    s = RecursiveCharacterTextSplitter(chunk_size=cfg["size"], chunk_overlap=cfg["overlap"])
    c = s.create_documents(docs[:3])
    r = retrieve(q, collection, c, k=cfg["k"])
    print(f"size={cfg['size']} overlap={cfg['overlap']} k={cfg['k']} -> {len(r)} chunks")
    if r: print(f"  first: {r[0][:60]}...")


size=100 overlap=10 k=1 -> 1 chunks
  first: Agentic AI systems use a perception-reasoning-action loop to...
size=200 overlap=20 k=2 -> 2 chunks
  first: Agentic AI systems use a perception-reasoning-action loop to...
size=500 overlap=50 k=5 -> 5 chunks
  first: Agentic AI systems use a perception-reasoning-action loop to...


### Gold — Query Rewriting in the Agent Loop

Integrate `rewrite_query` into the agent's search tool for better retrieval.

In [19]:
@tool
def smart_search(raw_query: str) -> str:
    """Search knowledge base with automatic query rewriting for better results."""
    rewritten = rewrite_query(raw_query)
    print(f"  Rewritten query: {rewritten}")
    return simple_rag(rewritten, docs, k=3)


gold_tools = [smart_search]

if not use_mock and llm is not None:
    agent = create_react_agent(
        model=llm,
        tools=gold_tools,
        prompt="You are a helpful assistant. Use tools when needed to provide accurate answers.",
    )
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Tell me about AI memory"}]}
    )
    final_msg = result["messages"][-1]
    print(final_msg.content)
else:
    print("Mock — agent with smart search:")
    print("  1. Rewrites vague query into specific search")
    print("  2. Searches knowledge base with rewritten query")
    print("  3. Returns most relevant information about AI memory")


/tmp/ipykernel_1423/1978136276.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


  Rewritten query: **Option 1: Comprehensive Technical Overview (Best for literature reviews/surveys)**
> **"Taxonomy of memory mechanisms in neural networks: parametric (weights), activation-based (RNN/LSTM/GRU hidden states), and external/non-parametric memory (Neural Turing Machines, MemN2N, Retrieval-Augmented Generation)"**

**Option 2: Architecture Comparison (Best for model selection/engineering)**
> **"Comparison of memory architectures in deep learning: Context window limits (Transformers) vs. Recurrent state compression (RNNs/LSTMs) vs. Retrieval-augmented memory (RAG, Memorizing Transformers) for long-sequence modeling"**

**Option 3: Specific Mechanism Deep-Dive (Best for implementation/theory)**
> **"How [specific mechanism, e.g., 'attention key-value caches' OR 'gating mechanisms in LSTMs/GRUs' OR 'differentiable neural computers'] function as memory in neural networks"**

**Option 4: Current SOTA / Applied Focus (Best for modern LLM contexts)**
> **"Long-term memory mech

## Next Steps

**Next notebook:** `03_planning.ipynb` — "The Agent's Brain"

> "Memory gives the agent context. Planning gives it direction."
